# OmniFold Publication Tools — Demo Notebook

This notebook demonstrates the prototype framework for organizing, validating,
and visualizing OmniFold outputs.

**Prerequisites:** Download the pseudodata HDF5 files from
[Zenodo (record 11507450)](https://zenodo.org/records/11507450)
and place them in `data/files/pseudodata/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.api import OmniFoldDataset
from src.schema import classify_columns
from weighted_histogram import weighted_histogram

## 1. Load the Dataset

In [ ]:
dataset = OmniFoldDataset('../data/files/pseudodata/')
print('Available files:', dataset.file_names)

In [ ]:
# Inspect the nominal file
summary = dataset.describe('multifold')
print(f"Events: {summary['n_events']}")
print(f"Total columns: {summary['n_columns']}")
print(f"Observables: {len(summary['columns']['observables'])}")
print(f"Weight columns: {len(summary['columns']['weights'])}")
print(f"Other columns: {len(summary['columns']['other'])}")
print()
print('Observable columns:', summary['columns']['observables'])

## 2. Visualize Key Observables with Weighted Histograms

In [ ]:
# Load nominal data
df = dataset.load('multifold')

# Dimuon pT distribution — the defining observable for this analysis
contents, edges, errors = weighted_histogram(
    df['pT_ll'].values,
    df['weights_nominal'].values,
    bins=np.linspace(200, 1000, 41),
    xlabel=r'$p_T^{\mu\mu}$ [GeV]',
    title='Dimuon Transverse Momentum (Nominal)',
    label='OmniFold nominal',
)

In [ ]:
# Compare nominal vs. MC weights for leading muon pT
weighted_histogram(
    df['pT_l1'].values,
    df['weights_nominal'].values,
    bins=np.linspace(100, 800, 36),
    xlabel=r'Leading muon $p_T$ [GeV]',
    title='Leading Muon pT: Nominal vs. MC',
    label='OmniFold nominal',
    comparison_weights=df['weight_mc'].values,
    comparison_label='MC weight',
)

## 3. Systematic Comparison: Nominal vs. Sherpa

In [ ]:
# Load the Sherpa systematic
df_sherpa = dataset.load('multifold_sherpa')

# Plot both on the same axes (normalized for fair comparison)
fig, ax = plt.subplots(figsize=(8, 6))
bins = np.linspace(200, 1000, 41)

h_nom, _ = np.histogram(df['pT_ll'], bins=bins, weights=df['weights_nominal'])
h_sherpa, _ = np.histogram(df_sherpa['pT_ll'], bins=bins, weights=df_sherpa['weights_nominal'])

# Normalize
h_nom = h_nom / h_nom.sum()
h_sherpa = h_sherpa / h_sherpa.sum()

centers = 0.5 * (bins[:-1] + bins[1:])
width = np.diff(bins)

ax.bar(centers, h_nom, width=width, alpha=0.4, label='Nominal (MG5)', color='steelblue')
ax.step(bins[:-1], h_sherpa, where='post', color='darkorange', linewidth=1.5, label='Sherpa')

ax.set_xlabel(r'$p_T^{\mu\mu}$ [GeV]', fontsize=13)
ax.set_ylabel('Normalized', fontsize=13)
ax.set_title('Generator Comparison: MG5 vs Sherpa', fontsize=14)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Ensemble Uncertainty Band

In [ ]:
# Compute histogram for each of the 100 ensemble replicas
bins = np.linspace(200, 1000, 41)
ensemble_cols = [c for c in df.columns if c.startswith('weights_ensemble_')]

ensemble_hists = np.array([
    np.histogram(df['pT_ll'], bins=bins, weights=df[col])[0]
    for col in ensemble_cols
])

# Nominal histogram
h_nom, _ = np.histogram(df['pT_ll'], bins=bins, weights=df['weights_nominal'])

# Uncertainty from ensemble spread (standard deviation across replicas)
ensemble_std = ensemble_hists.std(axis=0)

centers = 0.5 * (bins[:-1] + bins[1:])
width = np.diff(bins)

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(centers, h_nom, width=width, alpha=0.35, color='steelblue', label='Nominal')
ax.fill_between(
    centers, h_nom - ensemble_std, h_nom + ensemble_std,
    alpha=0.3, color='orange', label=r'Ensemble $\pm 1\sigma$',
    step='mid'
)
ax.set_xlabel(r'$p_T^{\mu\mu}$ [GeV]', fontsize=13)
ax.set_ylabel('Weighted events', fontsize=13)
ax.set_title('Dimuon pT with OmniFold Ensemble Uncertainty', fontsize=14)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Validation Suite

In [ ]:
# Run all validation checks
results = dataset.validate()
for r in results:
    print(r)

## 6. Column Classification

In [ ]:
# Show how columns are automatically classified
classified = classify_columns(df.columns.tolist())
print(f"Observables ({len(classified['observables'])}): {classified['observables']}")
print(f"\nWeight columns ({len(classified['weights'])}): first 10 = {classified['weights'][:10]}...")
print(f"\nOther ({len(classified['other'])}): {classified['other']}")